# 04 — Tokens, USD and wall-clock per success


In [ ]:
import sys, os
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve()))
from analysis._lib import (
    load_results, per_problem_pass_count, pass_at_k,
    bootstrap_pass_at_k, heatmap_matrix, mcnemar_pairs,
    REGISTRY, SYSTEMS, K,
)
results = load_results()
print(f'loaded {len(results)} attempt(s) across {len(set((r["system"], r["problem_id"]) for r in results))} (system, problem) cells')


In [ ]:
import json
prices = json.loads(open('analysis/prices.json').read())
print(f"{'system':<20}{'attempts':<10}{'successes':<10}{'tokens/success':<20}{'sec/success':<12}{'usd/success':<12}")
for sys in SYSTEMS:
    rs = [r for r in results if r['system'] == sys]
    ok = [r for r in rs if r.get('verdict')]
    total_tok = sum(int(r.get('tokens_in', 0)) + int(r.get('tokens_out', 0)) for r in rs)
    total_sec = sum(float(r.get('wall_seconds', 0)) for r in rs)
    price = prices.get(sys, {})
    in_p = price.get('input_per_mtok_usd') or 0
    out_p = price.get('output_per_mtok_usd') or 0
    total_usd = sum((int(r.get('tokens_in',0)) * in_p + int(r.get('tokens_out',0)) * out_p) / 1e6 for r in rs)
    tps = total_tok / max(1, len(ok))
    sps = total_sec / max(1, len(ok))
    ups = total_usd / max(1, len(ok))
    print(f'{sys:<20}{len(rs):<10}{len(ok):<10}{tps:<20.0f}{sps:<12.1f}{ups:<12.4f}')
